# Assignment: the integer program you do not have to solve as one

Four machines, four jobs, a cost for every pairing. Give each machine exactly one job and each job
exactly one machine, as cheaply as possible.

The obvious model uses a yes/no variable for each of the sixteen pairings. Sixteen binaries, eight
constraints, a small integer program. That is correct and it is also unnecessary, and the reason it
is unnecessary is the most useful thing in this notebook: **for this problem, the plain linear
program already returns whole numbers.** Asking for integrality changes nothing. The structure of
the constraints does the work the integrality would have done.

You will build the LP, watch it come back integral, build the binary version, watch it agree, and
then build the same problem a third way — as a transportation problem — and watch that agree too.

## Licence setup

In [1]:
import gurobipy as gp

env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)        # start silent: the licence banner, and its licence number, stay out of the outputs
try:
    from google.colab import userdata
    try:
        env.setParam("WLSACCESSID", userdata.get("GRB_WLSACCESSID"))
        env.setParam("WLSSECRET",   userdata.get("GRB_WLSSECRET"))
        env.setParam("LICENSEID",   int(userdata.get("GRB_LICENSEID")))
    except userdata.SecretNotFoundError:
        raise SystemExit("Add GRB_WLSACCESSID, GRB_WLSSECRET and GRB_LICENSEID as Colab Secrets "
                         "(key icon, left sidebar), then re-run this cell.")
    env.start()
    print("licence: Colab Secrets (WLS)")
except ImportError:
    env.start()
    print("licence: local gurobi.lic")

licence: local gurobi.lic


## The cost table

Sixteen numbers indexed by (machine, job). A table, read from `data/raw/`, shown as the matrix a
person thinks in.

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "src")))
from orteach import tolerance
from orteach.assignment import load_costs

cost = load_costs()
machines = sorted({i for i, _ in cost})
jobs = sorted({j for _, j in cost})

print(f"{'':10}" + "".join(f"{'job ' + j:>8}" for j in jobs))
for i in machines:
    print(f"machine {i:2} " + "".join(f"{cost[(i, j)]:8.0f}" for j in jobs))

             job 1   job 2   job 3   job 4
machine 1        14       5       8       7
machine 2         2      12       6       5
machine 3         7       8       3       9
machine 4         2       4       6      10


## Predict before building anything

There are 4 × 3 × 2 × 1 = 24 complete assignments. **Pick one by eye — the greedy one, say, giving
each machine its cheapest remaining job in order — and write down its total.** You will compare it
against the optimum and, more to the point, against your intuition about whether greedy is good
enough here.

In [3]:
greedy, taken, total = {}, set(), 0.0
for i in machines:
    j = min((j for j in jobs if j not in taken), key=lambda j: cost[(i, j)])
    greedy[i] = j
    taken.add(j)
    total += cost[(i, j)]
print("greedy assignment:", greedy)
print(f"greedy total: {total:.0f}")

greedy assignment: {'1': '2', '2': '1', '3': '3', '4': '4'}
greedy total: 20


## First model: the linear program

One variable per pairing, **continuous**, between 0 and 1. No integrality requested. If the solver
hands back 0.5 somewhere, that means half of machine 1 does job 2, which is meaningless — so the
question this model poses is whether that ever happens.

In [4]:
m = gp.Model(env=env)
tolerance.apply(m)
m.ModelSense = gp.GRB.MINIMIZE

x = m.addVars(cost.keys(), lb=0.0, ub=1.0, vtype=gp.GRB.CONTINUOUS, obj=cost, name="assign")
m.update()
print(f"{m.NumVars} continuous variables in [0, 1]")

16 continuous variables in [0, 1]


Two families of constraints. Each machine's row sums to one; each job's column sums to one.

In [5]:
m.addConstrs((x.sum(i, "*") == 1 for i in machines), name="machine")
m.addConstrs((x.sum("*", j) == 1 for j in jobs), name="job")
m.update()
print(f"{m.NumConstrs} constraints")
assert m.NumConstrs == len(machines) + len(jobs)

8 constraints


**Predict before solving.** Will any variable come back strictly between 0 and 1?

In [6]:
m.optimize()

lp_assign = {a: x[a].X for a in cost}
lp_pairs = sorted(a for a, v in lp_assign.items() if v > 0.5)

print(f"LP objective: {m.ObjVal:.1f}   (greedy was {total:.0f})\n")
for i, j in lp_pairs:
    print(f"  machine {i} -> job {j}   cost {cost[(i, j)]:.0f}")

LP objective: 15.0   (greedy was 20)

  machine 1 -> job 2   cost 5
  machine 2 -> job 4   cost 5
  machine 3 -> job 3   cost 3
  machine 4 -> job 1   cost 2


Check every one of the sixteen values, not just the ones that are on.

In [7]:
off_integer = [(a, v) for a, v in lp_assign.items() if min(abs(v), abs(v - 1)) > 1e-9]
print(f"values that are not 0 or 1: {len(off_integer)}")
for a, v in off_integer:
    print(f"  {a}: {v:.6f}")
assert not off_integer, "the LP relaxation returned a fraction - that should not happen here"
print("\nevery variable is exactly 0 or 1, and integrality was never requested")

values that are not 0 or 1: 0

every variable is exactly 0 or 1, and integrality was never requested


## Why the LP was enough

The constraint matrix of an assignment problem is **totally unimodular**: every square sub-matrix
has determinant 0, +1 or −1. For a matrix like that, with integer right-hand sides, every corner
point of the feasible region has integer coordinates — and the simplex method only ever stops at a
corner point. So the LP cannot return a fraction. It is not luck and it is not the solver being
clever; it is the shape of the constraints.

The same is true of the transportation problem, which is why the plans in the other notebook in
this folder came out whole when the data were whole.

## Second model: ask for binaries anyway

Same problem, `vtype=BINARY`. This is what the model looks like if you did not know the above.

In [8]:
m2 = gp.Model(env=env)
tolerance.apply(m2)
m2.ModelSense = gp.GRB.MINIMIZE
y = m2.addVars(cost.keys(), vtype=gp.GRB.BINARY, obj=cost, name="assign")
m2.addConstrs((y.sum(i, "*") == 1 for i in machines), name="machine")
m2.addConstrs((y.sum("*", j) == 1 for j in jobs), name="job")
m2.optimize()

ip_assign = {a: y[a].X for a in cost}
ip_pairs = sorted(a for a, v in ip_assign.items() if v > 0.5)
print(f"binary objective: {m2.ObjVal:.1f}")
print(f"same pairs as the LP: {ip_pairs == lp_pairs}")

binary objective: 15.0
same pairs as the LP: True


## Third model: it is a transportation problem

Give every machine a supply of one and every job a demand of one, and the assignment problem **is**
a transportation problem. The transportation solver in this package does not know anything about
assignments; hand it the instance and see what comes back.

In [9]:
from orteach.assignment import as_transportation
from orteach import transportation as tp

via = tp.solve(as_transportation(cost), env=env)
via_pairs = sorted(a for a, f in via.flow.items() if f > 0.5)
print(f"via transportation: {via.objective:.1f}")
print(f"same pairs: {via_pairs == lp_pairs}")

via transportation: 15.0
same pairs: True


Three formulations, one answer. When that happens it is worth asking which one to keep: the LP is
the smallest and the fastest, the binary version is the most honest about what the variables mean,
and the transportation view is the one that generalises — every result you know about
transportation problems now applies here.

---

# Now the streamlined version

`orteach.assignment` holds the LP and binary solves; the transportation view comes from the other
module, unchanged.

In [10]:
from orteach import assignment as asg
from orteach.tolerance import AGREEMENT_RTOL, rel_diff

pkg_lp = asg.solve_lp(cost, env=env)
pkg_ip = asg.solve_binary(cost, env=env)
pkg_via = tp.solve(asg.as_transportation(cost), env=env)

print(f"{pkg_lp.label:16} {pkg_lp.objective:6.1f}   integral: {pkg_lp.is_integral}")
print(f"{pkg_ip.label:16} {pkg_ip.objective:6.1f}")
print(f"{'via transport':16} {pkg_via.objective:6.1f}")

LP relaxation      15.0   integral: True
binary             15.0
via transport      15.0


## The agreement assertion

Three hand-built models against three package solves, objective and every one of the sixteen
variables each time.

In [11]:
checks = [("LP objective", m.ObjVal, pkg_lp.objective),
          ("binary objective", m2.ObjVal, pkg_ip.objective),
          ("transport objective", via.objective, pkg_via.objective)]
for a in cost:
    checks.append((f"LP x{a}", lp_assign[a], pkg_lp.assign[a]))
    checks.append((f"binary y{a}", ip_assign[a], pkg_ip.assign[a]))

worst = max(rel_diff(h, p) for _, h, p in checks)
print(f"{len(checks)} comparisons, worst relative difference {worst:.2e}")
assert worst < AGREEMENT_RTOL, f"notebook and package disagree by {worst:.2e}"
print(f"notebook and package agree to {worst:.1e}")

35 comparisons, worst relative difference 0.00e+00
notebook and package agree to 0.0e+00


---

## Where to take this next

- Add a fifth machine with no fifth job. The problem is no longer square. What is the right fix —
  a dummy job, or an inequality — and does the LP still come back integral?
- Change one cost so that two assignments tie for the optimum. Which one does the LP return, and is
  that answer stable if you reorder the rows of the table?
- Take the transportation notebook's DC-to-dealers instance and ask: is *its* constraint matrix
  totally unimodular? Then reconcile that with the fractional trucks it produced.